# Lab 1.3, Build 1 — Choose the model tier

Run the SAR-101 structured-summary task on both model tiers, 5 times each.
Record cost, latency, and field accuracy for every run.
Your seeded constraint is in `~/constraint.json` — it rules out at least one tier.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup ─────────────────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)

# Display the constraint
constraint = json.loads(pathlib.Path('/home/elastic/constraint.json').read_text())
print('=== Your seeded constraint ===')
print(f"  Label:       {constraint['label']}")
print(f"  Description: {constraint['description']}")
print()
print(f'FAST model:   {FAST}')
print(f'STRONG model: {STRONG}')

In [ ]:
# ── SAR-101 source text and task (do not edit) ────────────────────────────────
SAR_TEXT = """
Subject Name: Elias Vance
Account: Cortex Bank and Trust Premier Checking #4492-XXXX
Occupation: Freelance Management Consultant

Review Period: October 5, 2023 - October 18, 2023
Total Transaction Volume: $112,500
Suspicious Cash Withdrawals: $94,000
Typology: Structuring to evade Currency Transaction Report (CTR) requirements.

On October 5, 2023, Mr. Vance's account received a domestic wire transfer of $112,500
from Summit Holdings LLC. Between October 6 and October 15, 2023, Mr. Vance made daily
ATM withdrawals ranging from $9,000 to $9,500 per day, keeping each below the $10,000
CTR threshold. Total suspicious cash withdrawn: $94,000.
""".strip()

TASK_PROMPT = """Summarize this Suspicious Activity Report into a JSON object with exactly these four fields:
subject_name, account_number, suspicious_amount, typology.
Output only valid JSON, nothing else."""

SYSTEM_PROMPT = (
    "You are a compliance analyst assistant. When asked to summarize a SAR, "
    "output only valid JSON with the exact fields requested."
)

print('SAR-101 loaded. Task: summarize into 4-field JSON.')

---
## Fast tier — run 5 times

Edit the `# ── YOUR WORK ──` cell to make the API call. Then run the record cell once per run, incrementing the run number.

In [ ]:
# ── YOUR WORK ── Fast-tier call ───────────────────────────────────────────────
# Make one completion call. Store the result in `fast_response`.
# Don't change FAST — it reads ARA_MODEL_FAST from the environment.

# t0 = time.perf_counter()
# resp = client.chat.completions.create(
#     model=FAST,
#     messages=[
#         {"role": "system", "content": SYSTEM_PROMPT},
#         {"role": "user",   "content": f"{TASK_PROMPT}\n\n{SAR_TEXT}"},
#     ],
#     temperature=0,
# )
# latency_ms = (time.perf_counter() - t0) * 1000
# fast_response = {
#     'content': resp.choices[0].message.content,
#     'prompt_tokens':     resp.usage.prompt_tokens,
#     'completion_tokens': resp.usage.completion_tokens,
#     'latency_ms':        latency_ms,
# }

fast_response = {}  # ← replace with your call

In [ ]:
# Record this fast-tier run. Change RUN_NUM for each of your 5 runs (0 through 4).
RUN_NUM = 0   # ← change this each time you run

assert fast_response, "Run the fast-tier cell first."
trace = {
    'tier':        'fast',
    'model':       FAST,
    'run':         RUN_NUM,
    'latency_ms':  fast_response.get('latency_ms', 0),
    'response':    fast_response.get('content', ''),
    'prompt_tokens':     fast_response.get('prompt_tokens', 0),
    'completion_tokens': fast_response.get('completion_tokens', 0),
    'timestamp':   time.time(),
}
(TRACES / f'tier-fast-{RUN_NUM}.json').write_text(json.dumps(trace, indent=2))
print(f'Fast run {RUN_NUM} recorded. Response: {trace["response"][:120]}')

---
## Strong tier — run 5 times

Same task, different model. Run 5 times the same way.

In [ ]:
# ── YOUR WORK ── Strong-tier call ─────────────────────────────────────────────
# Same structure as the fast-tier call, but use STRONG as the model.

strong_response = {}  # ← replace with your call

In [ ]:
# Record this strong-tier run.
RUN_NUM_S = 0   # ← change this each time you run (0 through 4)

assert strong_response, "Run the strong-tier cell first."
trace_s = {
    'tier':        'strong',
    'model':       STRONG,
    'run':         RUN_NUM_S,
    'latency_ms':  strong_response.get('latency_ms', 0),
    'response':    strong_response.get('content', ''),
    'prompt_tokens':     strong_response.get('prompt_tokens', 0),
    'completion_tokens': strong_response.get('completion_tokens', 0),
    'timestamp':   time.time(),
}
(TRACES / f'tier-strong-{RUN_NUM_S}.json').write_text(json.dumps(trace_s, indent=2))
print(f'Strong run {RUN_NUM_S} recorded. Response: {trace_s["response"][:120]}')

---
## Summary — check your numbers before the Defend

In [ ]:
import statistics

GOLD_CHECKS = {
    'subject_name':     ['elias', 'vance'],
    'account_number':   ['4492'],
    'suspicious_amount':['94000', '94,000', '$94'],
    'typology':         ['structur'],
}

def field_accuracy(response_text):
    text = response_text.lower()
    hits = sum(1 for kws in GOLD_CHECKS.values() if any(kw in text for kw in kws))
    return hits / len(GOLD_CHECKS)

def summarize(tier):
    traces = sorted(TRACES.glob(f'tier-{tier}-*.json'))
    if not traces:
        print(f'  {tier}: no traces yet')
        return
    lats = [json.loads(t.read_text())['latency_ms'] for t in traces]
    accs = [field_accuracy(json.loads(t.read_text())['response']) for t in traces]
    print(f'  {tier}: {len(traces)} runs | p50 latency {statistics.median(lats):.0f} ms | '
          f'field accuracy {statistics.mean(accs):.0%}')

print('=== Tier summary ===')
summarize('fast')
summarize('strong')
print()

fast_count = len(list(TRACES.glob('tier-fast-*.json')))
strong_count = len(list(TRACES.glob('tier-strong-*.json')))
if fast_count >= 5 and strong_count >= 5:
    print('Both tiers have 5+ runs. Select Check in the sidebar.')
else:
    print(f'Need 5 runs each. Have: fast={fast_count}, strong={strong_count}.')